# 01: build patient-disjoint EyePACS splits (v2)

Fixes the leakage found in the v1 audit (2,033 patients shared between train and test).

**Image pool.** Keeps exactly the 19,332 images v1 retained, so the existing preprocessed
batches stay usable and nothing has to be re-cropped. The pool is inherited; only the
*assignment* to splits changes. The notebook characterises what v1 dropped so the exclusion
can be disclosed honestly even though the original filtering code is lost.

**Split rule.** Patient-disjoint 70/15/15, stratified on each patient's maximum grade
(so severe cases spread across splits), seed 20260920, fixed and recorded.

Writes to `dr_ensemble_v2/01_splits` and `dr_ensemble_v2/99_audit`.

In [5]:
# Cell 1: authorize and fetch source files
import io, os, json
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = build('drive', 'v3')

SRC = {
    'v1_train':  '1BD2WnBrCVfnDzkxmMwsSDjRX83EWtFbP',
    'v1_val':    '18E8theHeg_JC-9bOHZHiDZZO2Mol0Ap4',
    'v1_test':   '18p4IVE3UoBR-RRJuGaqXyW5nG2wr-VaZ',
    'labels':    '1DfCWmNV8raR-hXcjVx5noj3zHc2_Ui8t',
}
FOLDER_SPLITS = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
FOLDER_AUDIT  = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'

def fetch(fid, path):
    with io.FileIO(path, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid))
        done = False
        while not done:
            _, done = dl.next_chunk()

def upload(path, folder_id, mime='text/csv'):
    svc.files().create(
        body={'name': os.path.basename(path), 'parents': [folder_id]},
        media_body=MediaFileUpload(path, mimetype=mime), fields='id').execute()
    print('uploaded', os.path.basename(path))

for k, v in SRC.items():
    fetch(v, k + '.csv')
print('fetched:', {k: os.path.getsize(k + '.csv') for k in SRC})

fetched: {'v1_train': 208252, 'v1_val': 44615, 'v1_test': 44675, 'labels': 465317}


In [6]:
# Cell 2: characterise the inherited image pool (what v1 kept vs dropped)
import pandas as pd, numpy as np

labels = pd.read_csv('labels.csv')           # columns: image, level
v1 = pd.concat([pd.read_csv(f'v1_{s}.csv') for s in ['train', 'val', 'test']])

pool_ids = set(v1['image_id'].astype(str))
labels['image'] = labels['image'].astype(str)
labels['_kept'] = labels['image'].isin(pool_ids)

kept = labels[labels['_kept']].copy()
dropped = labels[~labels['_kept']].copy()

flow = {
    'source_labelled_images': int(len(labels)),
    'retained_images': int(len(kept)),
    'dropped_images': int(len(dropped)),
    'retained_fraction': round(len(kept) / len(labels), 4),
    'grade_dist_source':   {int(a): int(b) for a, b in labels['level'].value_counts().sort_index().items()},
    'grade_dist_retained': {int(a): int(b) for a, b in kept['level'].value_counts().sort_index().items()},
    'grade_dist_dropped':  {int(a): int(b) for a, b in dropped['level'].value_counts().sort_index().items()},
}
flow['retention_rate_by_grade'] = {
    g: round(flow['grade_dist_retained'].get(g, 0) / flow['grade_dist_source'][g], 4)
    for g in flow['grade_dist_source']
}
print(json.dumps(flow, indent=2))
print('\nIf retention_rate_by_grade is near-constant across grades, the v1 exclusion was')
print('roughly grade-independent and can be described as a random/quality subsample.')
print('If it varies a lot, v1 effectively rebalanced classes and that must be disclosed.')

{
  "source_labelled_images": 35126,
  "retained_images": 19332,
  "dropped_images": 15794,
  "retained_fraction": 0.5504,
  "grade_dist_source": {
    "0": 25810,
    "1": 2443,
    "2": 5292,
    "3": 873,
    "4": 708
  },
  "grade_dist_retained": {
    "0": 14232,
    "1": 1312,
    "2": 2919,
    "3": 489,
    "4": 380
  },
  "grade_dist_dropped": {
    "0": 11578,
    "1": 1131,
    "2": 2373,
    "3": 384,
    "4": 328
  },
  "retention_rate_by_grade": {
    "0": 0.5514,
    "1": 0.537,
    "2": 0.5516,
    "3": 0.5601,
    "4": 0.5367
  }
}

If retention_rate_by_grade is near-constant across grades, the v1 exclusion was
roughly grade-independent and can be described as a random/quality subsample.
If it varies a lot, v1 effectively rebalanced classes and that must be disclosed.


In [7]:
# Cell 3: build patient-disjoint stratified splits
SEED = 20260920
rng = np.random.default_rng(SEED)

df = kept[['image', 'level']].rename(columns={'image': 'image_id'}).copy()
df['patient_id'] = df['image_id'].str.split('_').str[0]
df['eye'] = df['image_id'].str.split('_').str[-1]
df['binary_label'] = (df['level'] >= 2).astype(int)

# one row per patient, stratum = that patient's worst eye
pat = df.groupby('patient_id')['level'].max().reset_index().rename(columns={'level': 'max_grade'})

assign = {}
for g, sub in pat.groupby('max_grade'):
    ids = sub['patient_id'].to_numpy()
    rng.shuffle(ids)
    n = len(ids)
    n_tr = int(round(0.70 * n))
    n_va = int(round(0.15 * n))
    for pid in ids[:n_tr]:            assign[pid] = 'train'
    for pid in ids[n_tr:n_tr + n_va]: assign[pid] = 'val'
    for pid in ids[n_tr + n_va:]:     assign[pid] = 'test'

df['split'] = df['patient_id'].map(assign)

# verification: hard failure if anything leaks
sets = {s: set(df.loc[df['split'] == s, 'patient_id']) for s in ['train', 'val', 'test']}
overlaps = {
    'train_val':  len(sets['train'] & sets['val']),
    'train_test': len(sets['train'] & sets['test']),
    'val_test':   len(sets['val'] & sets['test']),
}
assert sum(overlaps.values()) == 0, f'PATIENT LEAKAGE: {overlaps}'
assert len(df) == len(kept), 'image count changed'
print('patient-disjoint verified:', overlaps)

manifest = {
    'dataset': 'eyepacs', 'version': 'v2', 'seed': SEED,
    'split_rule': 'patient-disjoint 70/15/15, stratified on patient max grade',
    'image_pool': 'inherited from v1 retained set (19,332 images); assignment rebuilt',
    'binary_label_rule': 'referable = grade >= 2',
    'patient_overlap': overlaps,
    'n_images': {s: int((df['split'] == s).sum()) for s in ['train', 'val', 'test']},
    'n_patients': {s: int(len(sets[s])) for s in ['train', 'val', 'test']},
    'patients_with_both_eyes': {
        s: int((df[df['split'] == s].groupby('patient_id')['eye'].nunique() > 1).sum())
        for s in ['train', 'val', 'test']},
    'grade_dist': {
        s: {int(a): int(b) for a, b in df[df['split'] == s]['level'].value_counts().sort_index().items()}
        for s in ['train', 'val', 'test']},
    'referable_rate': {
        s: round(float(df[df['split'] == s]['binary_label'].mean()), 4)
        for s in ['train', 'val', 'test']},
    'sample_flow': flow,
}
print(json.dumps({k: v for k, v in manifest.items() if k != 'sample_flow'}, indent=2))

patient-disjoint verified: {'train_val': 0, 'train_test': 0, 'val_test': 0}
{
  "dataset": "eyepacs",
  "version": "v2",
  "seed": 20260920,
  "split_rule": "patient-disjoint 70/15/15, stratified on patient max grade",
  "image_pool": "inherited from v1 retained set (19,332 images); assignment rebuilt",
  "binary_label_rule": "referable = grade >= 2",
  "patient_overlap": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "n_images": {
    "train": 13532,
    "val": 2900,
    "test": 2900
  },
  "n_patients": {
    "train": 6766,
    "val": 1450,
    "test": 1450
  },
  "patients_with_both_eyes": {
    "train": 6766,
    "val": 1450,
    "test": 1450
  },
  "grade_dist": {
    "train": {
      "0": 9961,
      "1": 911,
      "2": 2053,
      "3": 340,
      "4": 267
    },
    "val": {
      "0": 2137,
      "1": 198,
      "2": 433,
      "3": 74,
      "4": 58
    },
    "test": {
      "0": 2134,
      "1": 203,
      "2": 433,
      "3": 75,
      "4": 55
    }
  

In [8]:
# Cell 4: write split files and manifest to Drive
cols = ['image_id', 'patient_id', 'eye', 'level', 'binary_label']
for s in ['train', 'val', 'test']:
    p = f'eyepacs_split_{s}_v2.csv'
    df.loc[df['split'] == s, cols].sort_values('image_id').to_csv(p, index=False)
    upload(p, FOLDER_SPLITS)

with open('eyepacs_splits_manifest_v2.json', 'w') as f:
    json.dump(manifest, f, indent=2)
upload('eyepacs_splits_manifest_v2.json', FOLDER_SPLITS, 'application/json')

with open('eyepacs_sample_flow_v2.json', 'w') as f:
    json.dump(flow, f, indent=2)
upload('eyepacs_sample_flow_v2.json', FOLDER_AUDIT, 'application/json')
print('\ndone')

uploaded eyepacs_split_train_v2.csv
uploaded eyepacs_split_val_v2.csv
uploaded eyepacs_split_test_v2.csv
uploaded eyepacs_splits_manifest_v2.json
uploaded eyepacs_sample_flow_v2.json

done
